In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import yaml
import sys
import importlib
from scipy.stats import ttest_ind

sys.path.append("../")
import functions as fn
importlib.reload(fn)



output_dir ="../data/clean/tests"
with open("../config.yaml") as file:
    config = yaml.safe_load(file)
config

In [ ]:
importlib.reload(fn)
df01 = pd.read_csv(config["output_data"]["file1"], delimiter = ",")
df02 = pd.read_csv(config["output_data"]["file2"], delimiter = ",", low_memory=False)
df = fn.merge_dfs_funct(df01, df02)

#df = df01.merge(df02, on = 'client_id', how = 'outer')
#display("df1 nunique client id = ", df1.client_id.nunique())
#display(df01.info())
#display(df02.info())
df.variation.unique()

In [ ]:
##############################################################################
# PLOT1: users with complete steps. NOT USED IN PRESENTATION
##############################################################################

def compl_steps_groups(df):
    """
    input: df cleaned (contains all merges)
    This function calculates the steps accomplished by each user
    output: adapt to output either the df or corresponding plot. By default it returns None
    """
    user_step_counts = df.groupby('client_id')['process_step'].nunique()

    step_distribution = user_step_counts.value_counts().sort_values(ascending = False)
    #step_distri_perc = user_step_counts.value_counts(normalize = True).sort_values(ascending = False)
    
    #plot:
    fig, axes = plt.subplots(figsize=(8,5))
    ax = sns.barplot(x = step_distribution.index,
            y = step_distribution.values,
            hue = step_distribution.index,
            legend = False,
            palette = ("Set3")
            )
    # Add labels on bars
    for container in ax.containers:
        ax.bar_label(container)
    
    plt.title("Distribution of Process Steps Completed")
    plt.xlabel("Number of Unique Steps Completed")
    plt.ylabel("Number of Users")
    plt.grid(True, axis='y', linestyle='--', alpha=0.7)
    #show numbers

    plt.show()
    return
clients_bars = compl_steps_groups(df)
clients_bars 
##############################################################################
# PLOT2 proportion of completed : NOT USED IN PRESENTATION
##############################################################################
def compl_steps_groups_pie(df):
    """
    input: df ceaned (contains all merges
    This function calculates the steps accomplished by each user
    output: pie chart of the proportions of the users that completed how number of steps
    """
    user_step_counts = df.groupby('client_id')['process_step'].nunique()
    step_distri_perc = user_step_counts.value_counts(normalize = True).sort_values(ascending = False)
    pie = step_distri_perc.plot.pie(autopct = '%1.1f%%', startangle=90, colors=sns.color_palette("Set3"))
    plt.show()
    return
clients_pie = compl_steps_groups_pie(df)

In [ ]:
##############################################################################
# Function:  user_id and number of completed steps: updated in functions.py: OK, NOT USED IN PRESENTATION
##############################################################################
def get_steps_by_client(df):
    """
    input: cleaned df or any processed df from original cleaned df
    Adds column at end with the number of steps completed by client
    """
    df1 = df.copy()
    df1 = df1.sort_values(['client_id', 'visit_id', 'date_time']).reset_index(drop = True)
    user_step_counts = (df1.groupby(['client_id', 'visit_id'])['process_step'].nunique().rename("n_of_unique_steps"))#.reset_index(drop = True)
    user_step_counts.reset_index()
    df1 = df1.merge(user_step_counts, on=['client_id','visit_id'] , how='left')
    df1["completed_steps"] = df1.n_of_unique_steps.apply(lambda x: "Yes" if x == 5 else "No")
    df1 = df1.sort_values(['client_id', 'visit_id', 'date_time']).reset_index(drop = True)
    return df1
#df.groupby(['client_id', 'visit_id'])['process_step'].nunique().rename("n_of_unique_steps")
df1 = get_steps_by_client(df)
df1[df1['client_id']== 9932367]

# 1. Steps accomplished by each group:
In the below charts, below, we can clearly that test groups performed better than the control group in accomplishing the 5 test steps, with around 10 units better: 55% vs 45%.
In the lower steps, it is the same tendency except for the start step, where control group accomplished only the starting step and didn't go beyond, which explicitly show that test group performed better in the tesing process.

In [ ]:
##############################################################################
# PLOT proportion of control/test _users in the completion: OK, NOT USED IN PRESENTATION
##############################################################################
def uniq_clnt_with_last_step_df(df):
    """
    RETURNS a df with the last step, for unique client_id
    """
    df1 = fn.get_steps_by_client(df)
    df1 = df1.sort_values(['client_id', 'visit_id', 'date_time']).reset_index(drop = True)
    keep_last_step_per_client = df1.groupby(['client_id', 'visit_id'])['n_of_unique_steps'].max().rename('last_reached_step')
    df1 = df1.merge(keep_last_step_per_client, on = ['client_id', 'visit_id'], how = 'right')
    df1 = df1.sort_values(['client_id', 'visit_id', 'date_time']).reset_index(drop = True)
    df1 = df1.drop_duplicates(subset = ['client_id'], keep = 'last', ignore_index = True)

    return df1
############################################################################## 
#Client groups by Steps: NOT USED in PRESENTAION
##############################################################################  
def clnt_groups_by_step_plot(df):
    """
    returns a bar plot with the proportion of test & control groups per step
    """
    df1 = fn.uniq_clnt_with_last_step_df(df)
    
    sns.set_theme(style="whitegrid") #use theme
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    df1 = df1[df1['variation'] != "Others"]   
    clnt_steps_perc = pd.crosstab(df1['n_of_unique_steps'], df1['variation'], normalize = 'index')*100
    clnt_steps_count = pd.crosstab(df1['n_of_unique_steps'], df1['variation'])
    
    clnt_steps_count.plot(kind = "bar", stacked = True, ax = axes[0])
    clnt_steps_perc.plot(kind = "bar", stacked = True, ax = axes[1])
    #display numbers
    ax_pct = axes[1]
    for container in ax_pct.containers:
        ax_pct.bar_label(
        container, fmt = '%.1f%%',
        label_type = 'center'
        )
    axes[0].set_xlabel('Accomplished Steps')
    axes[1].set_xlabel('Accomplished Steps')
    axes[0].set_ylabel('Number of Clients')
    axes[1].set_ylabel("Clients' Proportion")
    axes[0].set_title('Number of Clients per Accomplished Step')
    axes[1].set_title('Proportion of Clients per Accomplished Step')
    
    plt.show()
    
#df_compl_clnt = fn.get_steps_by_client(df)
df_compl_clnt = clnt_groups_by_step_plot(df)
df_compl_clnt

# Completion Mean Duration: Who is faster in completing all steps? 
## Comparison between Test group & Control group:
The Test group did slightly better from  control group, interms of completion speed: the boxplot suggests that Test group is slightly faster than Control group (5.32 min vs 5.35min), and both groups have many outliers corresponding to completion delay, sometimes exceeding 250 min
## Comparison between Female é Male speeds within the 2 groups:
The barchart shows that both males and females have nearly the same average of completion time, and it is around the average value (5.3 min).

In [ ]:
    
df_compl_duration = fn.get_compl_duration_df(df)

df_compl_duration.to_csv(f"{output_dir}/df_compl_duration.csv", index=False, encoding= "utf-8", sep = ";")

print("Mean compl duration for Control group: ",df_compl_duration[df_compl_duration['variation'] == "Control"]["all_steps_dur_min"].mean())
print("Mean compl duration for TEST group: ",df_compl_duration[df_compl_duration['variation']  == "Test"]["all_steps_dur_min"].mean())
print("Max compl duration for Control group: ",df_compl_duration[df_compl_duration['variation']  == "Control"]["all_steps_dur_min"].max()) 
print("Max compl duration for test group: ",df_compl_duration[df_compl_duration['variation']  == "Test"]["all_steps_dur_min"].max())
print("Min compl duration for Control group: ",df_compl_duration[df_compl_duration['variation']  == "Control"]["all_steps_dur_min"].min()) 
print("Min compl duration for test group: ",df_compl_duration[df_compl_duration['variation'] == "Test"]["all_steps_dur_min"].min())

# Step by Step duration:
The mean duration of steps increases when moving from a level to a higher one, suggesting probably that the degree of difficulty is increasing when porgressing: check the bar chart. Howver when the tendency stays the same for the two groups independently.

## Who is faster Males or females?
Females and males are running all over the steps in nearly the same speed. See chart, for both groups




In [ ]:
def get_step_by_step_duration_df(df):
    df1 = fn.get_process_duration(df)
    df1 = df1.sort_values(['client_id', 'visit_id', 'date_time']).reset_index(drop = True)
     
    return df1

df_step_by_step = get_step_by_step_duration_df(df)
#df_step_by_step.to_csv(f"{output_dir}/df_step_by_step.csv", index=False, encoding= "utf-8", sep = ";")
#df_step_by_step.shape #(302467, 21)

#df_step_by_step.head()

#df_step_by_step.to_csv(f"{output_dir}/df_step_by_step_groupby.csv", index=True, encoding= "utf-8", sep = ";")
df_step_by_step.tail(50)
#df_step_by_step.columns

# Duration testing Hyposthesis:OK
## Overall duration:
### Hypothesis definition:
We want to check whether we have enough evidence that the overl all step duration mean of test group is less than the mean of Control group. For this,
we define the following null hypothesis:
### Null Hypothesis: 
H0: test_duration_mean >= control_duration_mean. Mean duration of the test group is greater or equal than the mean duration of Control group.
H1: test_duration_mean < control_duration_mean.  Mean duration of the test group is less than the mean duration of Control group.
### Conclusion:
Fail to reject H0. The p_value:  1.00 > 0.05 there is enough evidence to say that the mean of Test group is less than the mean of Control group.

In [ ]:
##############################################################################
#H0: test_duration_mean = control_duration_mean.
#H1: test_duration_mean < control_duration_mean.
##############################################################################

    
test_hypo_duration = fn.run_test_hypo_duration(df)
test_hypo_duration


# Conversion and dropoff rates comparison:OK
Although the starting number of users of Test group is higher than that of Control users, the dropoff rates of control clients is faster, making the overall drop from start to confirm step as high as 34.04% against 29.96% for Test group. See Dashboard 'Dropoff Comparison'

# Where do users drop out? If not everyone completes all 5 steps
The funnel plot shows that the highest dropoff rate is between start and first step for Control users, suggesting that many users didnt complete the start step (~14%). However for Test users, the highest rate is recorded between step3 and confirmation step (8%), meaning that these users couldn't finalize the confirmation state.


In [ ]:
##############################################################################
# Define the Dropoff rates: funnel_df(df)
##############################################################################
funnel = fn.funnel_df(df)
#funnel.to_csv(f"{output_dir}/funnel_all_groups.csv", index=False, encoding= "utf-8", sep = ";")
funnel
funnel_bygroup = funnel.pivot(index = 'process_step', columns = 'variation', values = ['conversion_rate', 'dropoff_rate']).reset_index()
funnel_bygroup
#funnel_bygroup.to_csv(f"{output_dir}/funnel_variation.csv", index=False, encoding= "utf-8", sep = ";")
#funnel_bygroup.columns= ['process_step', 'control_group', 'Others_group','test_group',  'control_drop_off', 'others_drop_off', 'test_drop_off']
